# H&M LightGCN — seed50 나머지 5모형

**새 학습: M2 → M4 → M5_A(M2+M3) → M5_B(M2+M4) → M5_D(M2+M3+M4)**

현재 다른 GPU의 M1·M3·M5_C seed49/50 배치를 보완합니다. **이 노트북은 seed50만** 실행합니다. seed51·다른 백본·순열 대조군은 실행하지 않습니다.

- M1 seed50 완료 결과가 반드시 있어야 합니다. 없으면 새로 학습하지 않고 준비 전에 중단합니다.
- M3·M5_C seed50은 저장된 완료 결과가 있으면 읽어옵니다. 미완료라면 기다리거나 대신 학습하지 않습니다. 나중에 결과표 셀만 다시 실행하면 합쳐집니다.
- seed49 M2·M4 원본은 **확정 설계·입력 지문 확인용**입니다. seed50 성능표에 seed49 지표를 넣지 않습니다.
- 각 모델은 새 seed50 초기화와 하나의 optimizer로 공동학습합니다. 다른 모델의 checkpoint를 초기값으로 사용하거나 동결하지 않습니다.
- 입력·경제 특징·그래프는 한 번 준비해 공유합니다. **매 epoch Drive checkpoint 저장**, 완료 arm 재사용, 중단 재개를 유지합니다. 손상된 checkpoint는 삭제/자동 재학습하지 않고 중단합니다.

**조건 고정:** train 2018-09-20~2020-09-15 / test 2020-09-16~22, CLV 입력창 최근 학습365일, 300epoch 마지막 점에서만 test 평가. ID64·2층·batch131072·lr0.0005·ID L2 0.001·M2 공유 L2 1e-6·uniform 음성1개·MIN_ITEM_INTER=1. validation/holdout/조기종료 없음. 학습 구매쌍은 정답과 추천 후보에서 제외합니다.

이 결과는 **seed50 1시드 기술적 비교**이며 3/10시드 평균·SD·유의성 검정이 아닙니다. 이미 노출된 test에 맞춘 파라미터 수정·모델 선택은 하지 않습니다. 최종 시드 축소는 교수님과 별도 합의할 사항입니다.

빈 L4 GPU에서 실행하세요. 현재 H&M 배치와 GAT 런타임은 중단하거나 재시작하지 마세요. 동일 노트북을 두 GPU에서 동시에 실행하면 안 됩니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'L4 GPU 런타임을 연결하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU는 {GPU}입니다. 기존 조건과 같은 L4로 실행하세요.'
SOURCE_COMMIT = 'aa39a4be6f3f68b11ea53e003f2ffbdfb7e0f54e'
REPO = Path('/content/clv-hm-remaining5-s50-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)


## 1. 저장된 참조 확인 — 학습·데이터 전체 로드 없음

기본 경로는 기존 H&M M2 폴더와 M1/M3/M4/M5_C 폴더입니다. 기존 보고서가 갱신되어 원본 지문이 달라졌다면 **보관한 seed49 ZIP**을 `m2_reference`·`m4_reference`에 지정하세요. `core_reference`는 **M1 seed50의 arms/JSON이 있는 실행 루트 또는 그 상위 폴더**입니다. 평균 CSV로 대체하지 않습니다.

기본 새 결과: `/content/drive/MyDrive/논문/data/results_v3_hm_remaining5_lastweek_seed50_v1`. 중단 후 재개하려면 코드 pin·참조 경로·출력 경로를 그대로 유지하세요.

In [ ]:
import clv_hm_remaining5_lastweek_seed50 as screen
cfg = screen.configure(
    # m2_reference='/content/drive/MyDrive/clv_m2_adaptive_hm_seed49_lastweek_results.zip',
    # m4_reference='/content/drive/MyDrive/clv_hm_m4c_lastweek_seed49_results.zip',
    # core_reference='/content/drive/MyDrive/논문/data/results_v3_hm_m5c_lastweek_final10_v1',
)
print('seed49 M2 설계 확인:', cfg.m2_reference)
print('seed49 M4 설계 확인:', cfg.m4_reference)
print('seed50 M1/M3/M5_C 읽기:', cfg.core_reference)
print('새 결과/체크포인트:', cfg.out_dir)
refs = screen.load_references(cfg)
print('M1 seed50 저장 결과 확인 완료. 재학습·재평가하지 않습니다.')
print('새 학습 순서:', [screen.LABELS[m] for m in screen.MODELS])


## 2. 고정 입력 한 번 준비 — 아직 학습하지 않음

H&M 전체 입력 준비에는 시간이 걸립니다. M2 특징·M4 가중치 준비 시간과 M3 그래프 준비 시간을 출력합니다. 입력 지문·분할·M2 특징·M4 가중치·M3 진단이 기존 확정 결과와 다르면 학습 전에 중단합니다. M3 계수는 기존과 같은 학습 그래프 보정으로 산출하며 test 성능으로 선택하지 않습니다.

In [ ]:
prepared = screen.prepare(cfg)
assert cfg.seeds == (50,) and cfg.epochs == 300
assert prepared['data']['loss_w'] is None
assert set(prepared['data']['splits']) == {'test'}
print('실행 폴더:', prepared['run_dir'])
print('사전점검 완료. 다음 셀에서만 5모형 학습을 시작합니다.')


## 3. 다섯 모델 순차 학습 / 저장 epoch에서 재개

각 모델 300epoch 학습 후 최종 평가·보고서 저장을 마치면 다음 모델로 넘어갑니다. 완료 JSON이 있으면 재학습·재평가하지 않습니다. 미완료 모델은 마지막 정상 저장 epoch의 모델·optimizer·난수 상태에서 재개합니다. 중단된 epoch의 저장 전 부분은 다시 계산합니다.

학습 중에는 다른 셀을 실행하지 마세요. 재연결 후에는 위 준비 셀부터 같은 경로로 실행합니다. 런타임 자동 재연결을 보장하지는 않습니다. loss가 내려간다고 test 성능 개선이 확인된 것은 아닙니다.

In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
assert result['complete'], '아직 5모형이 모두 완료되지 않았습니다.'


## 4. 완료된 seed50 결과 비교 — 추가 학습·평가 없음

M1과 신규5모형은 이 배치의 필수 비교입니다. 다른 GPU에서 완료된 M3·M5_C seed50도 읽어오면 8모형 표가 됩니다. 아직 없는 열은 빈칸이며 실패가 아닙니다. 새 결과가 생기면 **이 셀만** 다시 실행하세요.

표는 12핵심지표를 표시합니다. CSV에는 전체142지표(전체·CLV 세그먼트·노출 포함), 모든 모형 쌍의 상대변화와 진단을 남깁니다. 동일 seed50 비교만 포함하며 평균·표준편차·CI를 산출하지 않습니다.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
order = ['M1', 'M2', 'M3', 'M4', 'M5_A', 'M5_B', 'M5_C', 'M5_D']
metrics = list(screen.CORE)
display(result['absolute'].set_index('model')[metrics].T.reindex(columns=order))
comparison = result['comparison']
vs_m1 = comparison[comparison['reference'].eq('M1')]
if not vs_m1.empty:
    display(vs_m1.pivot(index='metric', columns='model', values='relative_change_pct').reindex(metrics))
print('위 변화율은 M1 대비 %이며 통계적 유의성 판정이 아닙니다.')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))


## 5. 결과 ZIP 저장·다운로드

ZIP은 CSV/JSON/진단/원본 완료 arm만 담습니다. 큰 학습 checkpoint는 Drive의 progress 폴더에 그대로 보존합니다. **Drive 결과 폴더에도 ZIP을 먼저 복사**하므로 브라우저 다운로드가 멈춰도 Drive에서 내려받을 수 있습니다.

다섯 모델이 끝나기 전에 완료건만 회수하려면 `ALLOW_PARTIAL=True`로 바꾸세요. 이 값은 학습 조건이나 test 평가를 바꾸지 않으며, 부분 결과라는 표시를 유지합니다.

In [ ]:
from google.colab import files
import shutil, hashlib

ALLOW_PARTIAL = False
output = Path(screen.export_zip(prepared,
    '/content/hm_remaining5_lastweek_seed50_results.zip', allow_partial=ALLOW_PARTIAL))
destination = Path(cfg.out_dir) / output.name
shutil.copy2(output, destination)
assert hashlib.sha256(output.read_bytes()).digest() == hashlib.sha256(destination.read_bytes()).digest()
print('Drive ZIP 저장 완료:', destination)
print(f'크기: {output.stat().st_size / 1024**2:.2f} MB')
files.download(str(output))
